<a href="https://colab.research.google.com/github/NouhailaOULAARIF/recipe-graph-assistant/blob/main/Neo4j_Chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-generativeai
!pip install -q neo4j-driver
!pip install -q gradio

In [ ]:
import google.generativeai as gemini
import base64
import json
import gradio as gr
from neo4j import GraphDatabase
import re

In [ ]:
gemini.configure(api_key= "API_KEY")

In [ ]:
def get_answer(input):

    defaults = {
  'model': 'models/text-bison-001',
  'temperature': 0.7,
  'candidate_count': 1,
  'top_k': 40,
  'top_p': 0.95,
  'max_output_tokens': 1024,
  'stop_sequences': [],
  'safety_settings': [{"category":"HARM_CATEGORY_DEROGATORY","threshold":1},{"category":"HARM_CATEGORY_TOXICITY","threshold":1},{"category":"HARM_CATEGORY_VIOLENCE","threshold":2},{"category":"HARM_CATEGORY_SEXUAL","threshold":2},{"category":"HARM_CATEGORY_MEDICAL","threshold":2},{"category":"HARM_CATEGORY_DANGEROUS","threshold":2}],
    }

    prompt = f"""You are an expert in converting English questions to Neo4j Cypher Graph code! The Graph has following Node Labels - Recette , Étape , Ingrédient ,and the Recette Node label has the following properties - title, link, source, the Étape Node label has the following properties - Id and description, the  Ingrédient Node label has the following properties - nom, mesure and the Graph has the following Relationship types PREPARED_WITH_STEPS, CONTIENT!

    the relationships PREPARED_WITH_STEPS start from Recette Nodes to the Étape Nodes not the other way around and relationships CONTIENT start from Recette Nodes to the Ingrédient Nodes not the other way around.

    For example,
    Example 1 - Could you provide the ingredients for Carrot Bread, the Cypher command will be something like this
    ``` MATCH (Recette:Recette)-[:CONTIENT]->(Ingrédient:Ingrédient)
    WHERE Recette.title= 'Carrot Bread'
    RETURN Ingrédient.nom, Ingrédient.mesure
    ```

    Example 2 - How can I prepare the recipe for No-Bake Nut Cookies.
    ```
     MATCH (recette:Recette)-[:PREPARED_WITH_STEPS]->(etape:Étape)
    MATCH (recette)-[:CONTIENT]->(ingredient:Ingrédient)
    WHERE recette.title = 'No Bake Nut Cookies'
    WITH recette.title AS TitreRecette,
        COLLECT(DISTINCT etape.description) AS DescriptionsÉtapes,
        COLLECT(DISTINCT ingredient) AS Ingrédients
    RETURN TitreRecette,
          DescriptionsÉtapes,
          REDUCE(acc = "", i IN Ingrédients | acc + i.nom + " " + i.mesure + ", ") AS Ingrédients

    ```
    Example 3 - How can I prepare the recipe for Carrot Bread.
    ```
    MATCH (recette:Recette)-[:PREPARED_WITH_STEPS]->(etape:Étape)
    MATCH (recette)-[:CONTIENT]->(ingredient:Ingrédient)
    WHERE recette.title = 'Carrot Bread'
    WITH recette.title AS TitreRecette,
        COLLECT(DISTINCT etape.description) AS DescriptionsÉtapes,
        COLLECT(DISTINCT ingredient) AS Ingrédients
    RETURN TitreRecette,
          DescriptionsÉtapes,
          REDUCE(acc = "", i IN Ingrédients | acc + i.nom + " " + i.mesure + ", ") AS Ingrédients

    ```

    Example 4 - Provide the steps of the recipe No-Bake Nut Cookies.
    ```
    MATCH (recette:Recette)-[:PREPARED_WITH_STEPS]->(etape:Étape)
    WHERE recette.title = "No Bake Nut Cookies"
    RETURN etape.description AS DescriptionÉtape
    ORDER BY etape.Id

    ```
    Example 5 - can you give me recipes with flour as ingredient.
    ```
    MATCH (n:Recette)-[:CONTIENT]->(i:Ingrédient) WHERE i.nom="flour" RETURN DISTINCT n.title

    ```
    Example 6 - can you give me recipes with flour and sugar as ingredients.
    ```
    MATCH (r:Recette)-[:CONTIENT]->(i:Ingrédient)
    WHERE i.nom IN ['flour', 'sugar']
    WITH r, COUNT(i) AS ingredientsUsed
    RETURN r.title

    ```
     Example 7 - could you provide me the link of the recipe French Toast
    ```
     MATCH (r:Recette) WHERE r.title = "French Toast" RETURN r.link

    ```
    Dont include ``` and \n in the output

    {input}"""
    model = gemini.GenerativeModel('gemini-pro')
    response = model.generate_content(prompt)
    return response.text

In [ ]:
get_answer("Could you provide the ingredients for Carrot Bread?")

"MATCH (Recette:Recette)-[:CONTIENT]->(Ingrédient:Ingrédient)\nWHERE Recette.title= 'Carrot Bread'\nRETURN Ingrédient.nom, Ingrédient.mesure"

In [ ]:
driver = GraphDatabase.driver("bolt://VOTRE_ADRESSE_IP:7687", auth=("neo4j", "VOTRE_MOT_DE_PASSE"))

In [ ]:
def extract_query_and_return_key(input_query_result):
    slash_n_pattern = r'[ \n]+'
    ret_pattern = r'RETURN\s+(.*)'
    replacement = ' '

    cleaned_query = re.sub(slash_n_pattern, replacement, input_query_result)
    if cleaned_query:
        match = re.search(ret_pattern, cleaned_query)
        if match:
            extracted_string = match.group(1)
        else:
            extracted_string = ""
    return cleaned_query, extracted_string

In [ ]:
extract_query_and_return_key(get_answer("Could you provide the ingredients for Carrot Bread ?"))

("MATCH (Recette:Recette)-[:CONTIENT]->(Ingrédient:Ingrédient) WHERE Recette.title= 'Carrot Bread' RETURN Ingrédient.nom, Ingrédient.mesure",
 'Ingrédient.nom, Ingrédient.mesure')

In [ ]:
def format_names_with_ampersand(names):
    if len(names) == 0:
        return ""
    elif len(names) == 1:
        return names[0]
    else:
        formatted_names = ", ".join(names[:-1]) + " & " + names[-1]
        return formatted_names

In [ ]:
format_names_with_ampersand(["Carrot Bread","amal"])

'Carrot Bread & amal'

In [ ]:
def run_cypher_on_neo4j(inp_query):
    out_list = []
    with driver.session() as session:
        result = session.run(inp_query)
        for record in result:
            values = [str(value) for value in record.values()]
            out_list.append(", ".join(values))
    driver.close()
    return " & ".join(out_list)

In [ ]:
query_l, _ = extract_query_and_return_key(get_answer("Could you provide the ingredients for Cheeseburger Loaf?"))
run_cypher_on_neo4j(query_l)

' grated American cheese, 1 c. & ketchup, 1 Tbsp. & dry mustard, 1 tsp. & salt or to taste, 1 1/2 tsp. & finely chopped onion, 4 Tbsp. & ground chuck, 1 1/2 lb. & cracker crumbs, 1 c. & egg, 1 & evaporated milk, 1/2 c.'

In [ ]:
def generate_and_exec_cypher(input_query):
    gen_query, _ = extract_query_and_return_key(get_answer(input_query))
    return run_cypher_on_neo4j(gen_query)

In [ ]:
def chatbot(input, history=[]):
    output = str(generate_and_exec_cypher(input))
    history.append((input, output))
    return history, history

In [ ]:
gr.Interface(fn = chatbot,
             inputs = ["text",'state'],
             outputs = ["chatbot",'state']).launch(debug = True)

Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://a89d96ce934dd762ed.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


<ipython-input-30-bcfdde099f6c>:3: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:
